# Vallidate the Results of the Multi Unit-Regression 

The multi-unit regression uses building, parcel, and zillow data to calculate the units for multi-family homes when the provided unit is missing. To validate this data first we validate the output corresponds with the expected values. Then these values are validated at the census tract level to ensure the hosting capacity analysis is being completed on reliable data. 

In [2]:
# import necessary libraries
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt

In [20]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [7]:
# import Zillow data 
zillow = gpd.read_parquet("../../../../../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

# read in ca boundary to clip zillow
ca_boundary = gpd.read_file("../../../../../../capstone/electrigrid/data/ca_state_boundary.geojson").to_crs(epsg=4326)

# import multifamily homes with units 
multi_zillow_w_units = gpd.read_parquet("../../../../../../capstone/electrigrid/data/buildings/final_zillow_w_volume/multi_zillow_w_units.parquet").to_crs(epsg=4326)

# import multifamily homes with units 
keep_outliers_multi_zillow_w_units = gpd.read_parquet("../../../../../../capstone/electrigrid/data/buildings/final_zillow_w_volume/keep_outliers_multi_zillow_w_units.parquet").to_crs(epsg=4326)

# import single family homes with  adjusted units 
single_condo_zillow_w_units = gpd.read_parquet("../../../../../../capstone/electrigrid/data/buildings/final_zillow_w_volume/single_condo_zillow_w_units.parquet").to_crs(epsg=4326)

In [8]:
# clip Zillow to california 
zillow = zillow.clip(ca_boundary)

### Validate the outputs of the unit data with the original Zillow dataset

In [9]:
print(f"The orginal Zillow data includes ", zillow.shape[0], "points")

The orginal Zillow data includes  10012568 points


In [10]:
# investigate the unit totals
multi_zillow = zillow[(zillow['type'] == 'Multi') & (zillow['code'] != 'RR106')]
single_zillow = zillow[zillow['type'] == 'Single']
condo_multi = zillow[(zillow['type'] == 'Multi') & (zillow['code'] == 'RR106')]

In [12]:
# condo data is crazy!
zillow[(zillow['type'] == 'Multi') & (zillow['code'] == 'RR106')].head(3)

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (-115.48046 32.67979)
8104552,Multi,NaN,NaN,fossil,central,O,NaN,229500.0,living,1452.0,1429873,06025012003,h279,Others,RR106,POINT (-115.48046 32.67979)
8104601,Multi,NaN,NaN,fossil,central,None,NaN,159291.0,living,1276.0,1429922,06025012003,h279,Others,RR106,POINT (-115.48046 32.67979)


In [13]:
print(f"There are ", multi_zillow['unit'].sum() + single_zillow.shape[0] + condo_multi.shape[0], "units before the regression of multifamily homes.")

There are  13107697.0 units before the regression of multifamily homes.


In [14]:
print(f"The regressed Zillow data is", multi_zillow_w_units.shape[0] + single_condo_zillow_w_units.shape[0] ,"points")

The regressed Zillow data is 9940002 points


In [15]:
print(f"The number of homes calculated in California by unit (when dropping outliers):", single_condo_zillow_w_units['total_unit'].sum() + multi_zillow_w_units['total_unit'].sum())

The number of homes calculated in California by unit (when dropping outliers): 20473622.0


In [16]:
print(f'The number of homes calculated in California by unit (when keeping outliers): ', single_condo_zillow_w_units['total_unit'].sum() + keep_outliers_multi_zillow_w_units['total_unit'].sum())

The number of homes calculated in California by unit (when keeping outliers):  12505709.0


# Investigate differences

In [17]:
print(f"Maximum units for multi-family homes before the regression",multi_zillow['unit'].max())

Maximum units for multi-family homes before the regression 6857.0


In [18]:
print(f"Maximum units for the multi-family homes after the regression", multi_zillow_w_units['total_unit'].max())

Maximum units for the multi-family homes after the regression 32508.0


In [22]:
multi_zillow_w_units[multi_zillow_w_units['total_unit'] > 20000]

,parcel_ID,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,area_m2,volume_m3,total_volume_m3,geometry
70056,1741158,osm,1076648548,4.239654,1.235935,USA,"{'xmax': -121.32035999999997, 'xmin': -121.320...",Multi,NaN,NaN,fossil,central,None,18857590.0,living,203928.0,5412257,06061021403,80,PGE/SCE,RI103,3793394,25898.0,277.320948,1175.744752,1.873520e+06,POINT (-121.31155 38.88203)
70115,1741991,ms,UnitedStates_023010033_48507,7.166322,0.922174,USA,"{'xmax': -121.07778351764603, 'xmin': -121.078...",Multi,NaN,2.0,elec,None,None,635342.0,living,4612.0,5393461,06061020300,612,PGE/SCE,RI103,3783944,32508.0,602.098473,4314.831385,1.800838e+06,POINT (-121.06943 38.91169)
70151,1742546,ms,UnitedStates_023010033_40271,2.930069,0.692705,USA,"{'xmax': -121.09698268310447, 'xmin': -121.097...",Multi,NaN,2.0,elec,None,O,908431.0,living,3217.0,5423464,06061021501,341,PGE/SCE,RI102,3802682,21978.0,139.837178,409.732608,4.993344e+05,POINT (-121.08896 38.92205)


From code, know they are triplexes (RI102) and quadruplexes (RI103)

Explore the number of units in the original Zillow df

In [27]:
zillow[zillow.index.isin([3793394, 3783944, 3802682])]

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry
3783944,Multi,NaN,2.0,elec,None,None,4.0,635342.0,living,4612.0,5393461,06061020300,612,PGE/SCE,RI103,POINT (-121.06986 38.90460)
3802682,Multi,NaN,2.0,elec,None,O,4.0,908431.0,living,3217.0,5423464,06061021501,341,PGE/SCE,RI102,POINT (-121.08477 38.91939)
3793394,Multi,NaN,NaN,fossil,central,None,240.0,18857590.0,living,203928.0,5412257,06061021403,80,PGE/SCE,RI103,POINT (-121.30563 38.88513)


Look slike the quadruplexes were accurate, whereas the triplex was not?

In [28]:
multi_zillow_w_units[multi_zillow_w_units['total_unit'] > 10000]

,parcel_ID,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,area_m2,volume_m3,total_volume_m3,geometry
38977,543106,osm,582556168,4.224887,0.281700,USA,"{'xmax': -120.4548948, 'xmin': -120.4551705999...",Multi,NaN,NaN,None,None,I,9736.0,living,4320.0,580321,06009000122,129,PGE/SCE,RI109,495938,16464.0,236.116360,997.564915,9.975649e+02,POINT (-120.45512 38.13823)
66969,1609804,ms,UnitedStates_023012130_128213,4.362795,1.790771,USA,"{'xmax': -119.07907524101593, 'xmin': -119.079...",Multi,2022.0,5.0,others,others,None,427133.0,living,2109.0,1718760,06029003220,147,PGE/SCE,RI101,1440374,15158.0,1874.664320,8178.775889,3.337407e+04,POINT (-119.08108 35.28530)
69804,1729868,ms,UnitedStates_023010211_317363,5.165205,0.781673,USA,"{'xmax': -121.00026477736984, 'xmin': -121.000...",Multi,1979.0,5.0,None,None,None,1945548.0,living,9408.0,1002152,06017030810,678,PGE/SCE,RI103,861990,18848.0,292.316774,1509.875924,2.316813e+06,POINT (-121.00006 38.69217)
70056,1741158,osm,1076648548,4.239654,1.235935,USA,"{'xmax': -121.32035999999997, 'xmin': -121.320...",Multi,NaN,NaN,fossil,central,None,18857590.0,living,203928.0,5412257,06061021403,80,PGE/SCE,RI103,3793394,25898.0,277.320948,1175.744752,1.873520e+06,POINT (-121.31155 38.88203)
70115,1741991,ms,UnitedStates_023010033_48507,7.166322,0.922174,USA,"{'xmax': -121.07778351764603, 'xmin': -121.078...",Multi,NaN,2.0,elec,None,None,635342.0,living,4612.0,5393461,06061020300,612,PGE/SCE,RI103,3783944,32508.0,602.098473,4314.831385,1.800838e+06,POINT (-121.06943 38.91169)
70151,1742546,ms,UnitedStates_023010033_40271,2.930069,0.692705,USA,"{'xmax': -121.09698268310447, 'xmin': -121.097...",Multi,NaN,2.0,elec,None,O,908431.0,living,3217.0,5423464,06061021501,341,PGE/SCE,RI102,3802682,21978.0,139.837178,409.732608,4.993344e+05,POINT (-121.08896 38.92205)
70165,1742770,ms,UnitedStates_023010033_283,6.405213,1.203548,USA,"{'xmax': -121.09756670130248, 'xmin': -121.097...",Multi,NaN,3.0,fossil,central,None,465000.0,living,2227.0,5423175,06061021501,575,PGE/SCE,RI101,3802438,12257.0,573.526316,3673.558149,9.678237e+05,POINT (-121.08808 38.92943)
70507,1752557,osm,825933825,11.080114,1.921150,USA,"{'xmax': -121.060198, 'xmin': -121.060348, 'ym...",Multi,NaN,7.0,None,None,None,395000.0,living,3784.0,4519994,06057000602,81,PGE/SCE,RI104,3145029,13005.0,97.263434,1077.689976,1.913469e+06,POINT (-121.06990 39.21575)
315641,5771986,osm,270490185,7.112334,1.765728,USA,"{'xmax': -117.1681442, 'xmin': -117.168349, 'y...",Multi,2004.0,NaN,fossil,central,None,954508.0,None,NaN,5898085,06065050702,1208,PGE/SCE,RI104,4102993,15714.0,486.073106,3457.114402,9.779628e+03,POINT (-117.16857 33.63411)
323358,6243525,osm,270490161,8.857489,4.040596,USA,"{'xmax': -117.16821460000001, 'xmin': -117.168...",Multi,2004.0,NaN,fossil,central,None,736904.0,None,NaN,5898096,06065050702,1208,PGE/SCE,RI104,4103004,10864.0,507.328874,4493.659737,4.493660e+03,POINT (-117.16834 33.63302)


In [30]:
zillow[zillow.index.isin([13824.0])]

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry
13824,Single,1910.0,2.0,None,None,I,1.0,1564761.0,living,1078.0,15452,06001400300,574,PGE/SCE,RR101,POINT (-122.25757 37.83831)


In [32]:
zillow[zillow.index.isin([5650504])]

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry
5650504,Multi,NaN,2.0,fossil,None,None,24.0,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,POINT (-122.26084 37.50303)


In [33]:
multi_zillow_w_units[multi_zillow_w_units['zillow_index'] == 5650504]

,parcel_ID,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,area_m2,volume_m3,total_volume_m3,geometry
465780,9246371,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465781,9246372,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465782,9246373,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465783,9246374,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465784,9246375,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465785,9246376,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465786,9246377,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465787,9246378,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465788,9246379,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)
465789,9246380,osm,385951507,6.916929,1.941668,USA,"{'xmax': -122.2605461, 'xmin': -122.2607557999...",Multi,NaN,2.0,fossil,None,None,478258.0,living,1425.0,9210127,06081609201,165,PGE/SCE,RI112,5650504,13824.0,169.637169,1173.368294,14280.053673,POINT (-122.26091 37.50295)


This is a single family home...why did we predict units for it in the first place?